In [ ]:
# conda create --name qwen --clone ovis
# conda activate qwen
# module load cuda/12.2
# pip install qwen-vl-utils==0.0.8
# pip install torchvision==0.19.0
# pip install gekko zstandard
# pip install git+https://github.com/casper-hansen/AutoAWQ.git --no-deps

# pip install autoawq --no-deps

In [ ]:
import re
import csv
import glob
import json
import torch

from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-32B-Instruct-AWQ",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-32B-Instruct-AWQ")

In [ ]:
text = """You are an image analysis tool specialized in facial attribute classification. 
          For the provided face image, output a JSON object with the following attributes:
        
            {
              "gender": ["male", "female"],
              "age": ["young", "middle-aged", "senior"],
              "skin_color": ["light", "medium", "dark"],
              "ancestry": ["asian", "south_asian", "black", "latino/hispanic", "middle_eastern", "white", "indigenous"],
              "hair_color": ["black", "brown", "red", "blonde", "gray", "other"],
              "bangs": ["yes", "no"],
              "bald": ["yes", "no"],
              "beard": ["no", "mustache", "stubble", "full"],
              "glasses": ["no", "regular", "sun"],
              "headwear": ["no", "beanie", "cap", "hat", "headband", "hijab", "helmet", "turban"],
            }

          Ensure the labeling is based on visible evidence only. If an attribute is unclear, return "unknown".
          
          Only output the JSON without any additional explanation or text.
          
          Example JSON output:
          
          {
            "gender": "female",
            "age": "middle-aged",
            "skin_color": "light",
            "ancestry": "asian",
            "hair_color": "black",
            "bangs": "no",
            "bald": "no",
            "beard": "no",
            "glasses": "sun",
            "headwear": "beanie",
          }
          """

In [ ]:
default_keys = ["gender", "age", "skin_color", "ancestry", "hair_color", "bangs", 
                "bald", "beard", "glasses", "headwear"]

# Function to process a single image
def process_images(batch_images):
    # Sample messages for batch inference
    messages = []
    for image_path in batch_images:
        messages.append([
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": text},
                ],
            }
        ])
    # Preparation for batch inference
    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages
    ]
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_texts = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    result = []
    for i in range(len(batch_images)):
        # Extracting the JSON part
        json_output = re.search(r"\{.*\}", output_texts[i], re.DOTALL).group()
        json_output = re.sub(r',\s*}', '}', json_output)  # This removes a comma just before a closing curly brace
        data = json.loads(json_output)
        result.append([batch_images[i].replace('../../../data/RFW/data/', '')] + [data.get(key, "unknown") for key in default_keys])
    torch.cuda.empty_cache()
    return result

In [ ]:
init = 0
files = glob.glob('../../../data/RFW/data/*/*/*')

import os
os.makedirs("../../../results/annotations", exist_ok=True)
output_file = "../../../results/annotations/qwen_labels.csv"

# Column names
columns = ["File Name", "Gender", "Age", "Skin Color", "Ancestry", "Hair Color", 
           "Bangs", "Bald", "Beard", "Glasses", "Headwear"]

# Chunk settings
chunk_size = 32  # Number of images processed per batch

# Write CSV header
with open(output_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)  # CSV Header

# Process in chunks
for start_idx in tqdm(range(init, len(files), chunk_size), desc="Processing Images"):
    end_idx = min(start_idx + chunk_size, len(files))
    chunk = files[start_idx:end_idx]
    results = process_images(chunk)
    # Save chunk results to CSV
    with open(output_file, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)